# Bacterial Taxonomy Classification with SILVA

This notebook uses outputs from `Qiime2_Pipeline_v2.ipynb` to classify bacterial ASVs with a SILVA Naive Bayes classifier.

## 1. Paths and Inputs

Set the sample inputs, SILVA training files, and output locations. Make sure to run this even if the classifier has been trained.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import qiime2
import shutil
import subprocess
import textwrap

work_dir = Path.cwd()
if not (work_dir / "Qiime2_Taxonomy_Classification_SILVA.ipynb").exists():
    candidate = Path("/Users/clarence/Savolainen Lab/Microbiome/Qiime")
    if (candidate / "Qiime2_Taxonomy_Classification_SILVA.ipynb").exists():
        work_dir = candidate

output_dir = work_dir / "output"
silva_dir = work_dir / "SILVA classifier"
decontam_dir = output_dir / "decontam"
decontam_test_dir = decontam_dir / "three_samples"
decontam_full_dir = decontam_dir / "full"

table_path = output_dir / "feature_table.qza"
rep_seqs_path = output_dir / "rep_seqs.qza"
metadata_path = output_dir / "metadata.tsv"

three_sample_manifest_path = work_dir / "manifest_test_3samples.tsv"
full_manifest_path = work_dir / "manifest.tsv"
negative_controls_path = work_dir / "Vincent_Negative_Controls.csv"
dna_con_path = work_dir / "DNA_Con_Vincent.csv"

silva_fasta = silva_dir / "SILVA_138.2_SSURef_NR99_tax_silva.fasta"
silva_gz = silva_dir / "SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz"
fixed_fasta = silva_dir / "SILVA_fixed.fasta"
silva_tax_tsv = silva_dir / "silva-taxonomy.tsv"
silva_seqs_qza = silva_dir / "silva-seqs.qza"
silva_tax_qza = silva_dir / "silva-tax.qza"
ref_reads_qza = silva_dir / "ref-seqs.qza"
classifier_path = silva_dir / "silva-trained-classifier.qza"

decontam_test_filtered_table_qza = decontam_test_dir / "feature_table_decontam.qza"
decontam_test_filtered_rep_seqs_qza = decontam_test_dir / "rep_seqs_decontam.qza"
decontam_test_metadata_path = decontam_test_dir / "decontam_metadata.tsv"
decontam_test_taxonomy_out = decontam_test_dir / "taxonomy_silva.qza"
decontam_test_barplot_out = decontam_test_dir / "taxa_barplot_silva.qzv"
decontam_test_export_dir = decontam_test_dir / "export"
decontam_test_export_dir_taxonomy = decontam_test_dir / "taxonomy_export"
decontam_test_biom_path = decontam_test_dir / "data_feature-table.biom"
decontam_test_filtered_biom_path = decontam_test_dir / "filtered_feature-table.biom"
decontam_test_contaminants_path = decontam_test_dir / "contaminant_features.tsv"
decontam_test_filtered_fasta = decontam_test_dir / "rep_seqs_decontam.fasta"
decontam_test_rep_export_dir = decontam_test_dir / "rep_seqs_export"
decontam_test_aligned_seqs = decontam_test_dir / "aligned_rep_seqs.qza"
decontam_test_masked_aligned_seqs = decontam_test_dir / "masked_aligned_rep_seqs.qza"
decontam_test_unrooted_tree = decontam_test_dir / "unrooted_tree.qza"
decontam_test_rooted_tree = decontam_test_dir / "rooted_tree.qza"

decontam_full_filtered_table_qza = decontam_full_dir / "feature_table_decontam.qza"
decontam_full_filtered_rep_seqs_qza = decontam_full_dir / "rep_seqs_decontam.qza"
decontam_full_metadata_path = decontam_full_dir / "decontam_metadata.tsv"
decontam_full_taxonomy_out = decontam_full_dir / "taxonomy_silva.qza"
decontam_full_barplot_out = decontam_full_dir / "taxa_barplot_silva.qzv"
decontam_full_export_dir = decontam_full_dir / "export"
decontam_full_export_dir_taxonomy = decontam_full_dir / "taxonomy_export"
decontam_full_biom_path = decontam_full_dir / "data_feature-table.biom"
decontam_full_filtered_biom_path = decontam_full_dir / "filtered_feature-table.biom"
decontam_full_contaminants_path = decontam_full_dir / "contaminant_features.tsv"
decontam_full_filtered_fasta = decontam_full_dir / "rep_seqs_decontam.fasta"
decontam_full_rep_export_dir = decontam_full_dir / "rep_seqs_export"
decontam_full_aligned_seqs = decontam_full_dir / "aligned_rep_seqs.qza"
decontam_full_masked_aligned_seqs = decontam_full_dir / "masked_aligned_rep_seqs.qza"
decontam_full_unrooted_tree = decontam_full_dir / "unrooted_tree.qza"
decontam_full_rooted_tree = decontam_full_dir / "rooted_tree.qza"

print(f"Working directory: {work_dir}")
print(f"Output directory: {output_dir}")
print(f"SILVA directory: {silva_dir}")
print(f"Three-sample decontam directory: {decontam_test_dir}")
print(f"Full decontam directory: {decontam_full_dir}")

Working directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime
Output directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output
SILVA directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime/SILVA classifier
Three-sample decontam directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples
Full decontam directory: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full


## 2. Validate Required Inputs

Check the ASV outputs, metadata file, and SILVA source file before running any classification steps.

In [2]:
required = [rep_seqs_path, table_path, metadata_path]
missing = [str(p) for p in required if not p.exists()]

if missing:
    print("Missing required files:")
    for p in missing:
        print(f"  - {p}")
    raise FileNotFoundError("Add the missing files before continuing.")

silva_source = silva_fasta if silva_fasta.exists() else silva_gz if silva_gz.exists() else None
if silva_source is None:
    raise FileNotFoundError("Add the SILVA FASTA or FASTA.GZ file before continuing.")

print("All required inputs found.")
print(f"Using SILVA source: {silva_source}")

All required inputs found.
Using SILVA source: /Users/clarence/Savolainen Lab/Microbiome/Qiime/SILVA classifier/SILVA_138.2_SSURef_NR99_tax_silva.fasta


## 2.5. Fix SILVA RNA Bases

Convert U bases to T in the SILVA FASTA so QIIME 2 can use it as a DNA reference.

In [ ]:
import gzip

if not fixed_fasta.exists():
    print("Fixing RNA bases (U -> T)...")

    silva_source = silva_fasta if silva_fasta.exists() else silva_gz if silva_gz.exists() else None
    if silva_source is None:
        raise FileNotFoundError("SILVA FASTA or FASTA.GZ not found. Check section 1 (Paths and Inputs).")

    open_func = open
    if silva_source.suffix == '.gz':
        open_func = gzip.open

    with open_func(silva_source, 'rt') as fin, open(fixed_fasta, 'w') as fout:
        for line in fin:
            if line.startswith('>'): # keeps header lines unreplaced
                fout.write(line)
            else:
                fout.write(line.replace('U', 'T'))

print(f"Using corrected SILVA FASTA: {fixed_fasta}")


Using corrected SILVA FASTA: /Users/clarence/Savolainen Lab/Microbiome/Qiime/SILVA classifier/SILVA_fixed.fasta


## 3. Build the SILVA Reference Artifacts

Convert the SILVA FASTA headers into a taxonomy TSV and import the reference sequences and taxonomy into QIIME 2.

In [25]:
import subprocess
import qiime2
from qiime2.plugins import feature_classifier

if ref_reads_qza.exists():
    print(f"✓ SILVA reference artifacts already built: {ref_reads_qza}")
else:
    if not silva_tax_tsv.exists():
        print("Building SILVA taxonomy TSV from FASTA headers...")
        with open(fixed_fasta, 'r') as fin, open(silva_tax_tsv, 'w') as fout:
            for line in fin:
                if line.startswith('>'):
                    header = line[1:].strip()
                    parts = header.split(' ', 1)
                    seq_id = parts[0]
                    taxonomy = parts[1].split(' ', 1)[0] if len(parts) > 1 else ''
                    fout.write(f"{seq_id}\t{taxonomy}\n")

    if not silva_seqs_qza.exists():
        print("Importing SILVA reference sequences...")
        subprocess.run([
            'qiime', 'tools', 'import',
            '--type', 'FeatureData[Sequence]',
            '--input-path', str(fixed_fasta),
            '--output-path', str(silva_seqs_qza),
        ], check=True)

    if not silva_tax_qza.exists():
        print("Importing SILVA taxonomy...")
        subprocess.run([
            'qiime', 'tools', 'import',
            '--type', 'FeatureData[Taxonomy]',
            '--input-format', 'HeaderlessTSVTaxonomyFormat',
            '--input-path', str(silva_tax_tsv),
            '--output-path', str(silva_tax_qza),
        ], check=True)

    silva_seqs = qiime2.Artifact.load(str(silva_seqs_qza))
    silva_tax = qiime2.Artifact.load(str(silva_tax_qza))

    ref_reads = feature_classifier.methods.extract_reads(
        sequences=silva_seqs,
        f_primer='CCTACGGGNGGCWGCAG',
        r_primer='GACTACHVGGGTATCTAATCC',
        trunc_len=0
    ).reads

    ref_reads.save(str(ref_reads_qza))
    print(f"Saved reference reads: {ref_reads_qza}")


✓ SILVA reference artifacts already built: /Users/clarence/Savolainen Lab/Microbiome/Qiime/SILVA classifier/ref-seqs.qza


## 4. Train a Fresh SILVA Classifier

Train a Naive Bayes classifier on the V3-V4 SILVA reference. This step is skipped if a trained classifier already exists. On subsequent runs, you can skip sections 2.5-4 and continue with section 5A/5B (decontam) or section 6 (classification).

In [26]:
from qiime2.plugins import feature_classifier

if classifier_path.exists():
    print(f"Classifier already trained: {classifier_path}")
    print("Skipping training. Proceed to section 5A/5B (decontam) or section 6 (classify ASVs).")
else:
    print("Training SILVA Naive Bayes classifier...")
    classifier = feature_classifier.methods.fit_classifier_naive_bayes(
        reference_reads=ref_reads,
        reference_taxonomy=silva_tax
    ).classifier

    classifier.save(str(classifier_path))
    print(f"Saved classifier: {classifier_path}")


Classifier already trained: /Users/clarence/Savolainen Lab/Microbiome/Qiime/SILVA classifier/silva-trained-classifier.qza
Skipping training. Proceed to section 5A/5B (decontam) or section 6 (classify ASVs).


## 5. Remove Contaminants with decontam (R)

Run an R decontam workflow on the feature table before taxonomy classification. This section is split into two parts:

- 5A: a three-sample test run so you can confirm the blank and concentration metadata are wired correctly.
- 5B: the full run for the complete sample set once the test run looks right.

Both runs use the blank fastq folder, the DNA concentration file, and the negative controls manifest to build the decontam metadata table.

Before running either block, make sure R, phyloseq, and decontam are available in the environment.

In [ ]:
import numpy as np
import pandas as pd
import qiime2
import shutil
import subprocess
import textwrap
from qiime2.plugins import feature_table


def _orient_feature_table_df(table_df, run_sample_ids):
    table_df = table_df.copy()
    table_df.index = table_df.index.astype(str)
    table_df.columns = table_df.columns.astype(str)

    run_sample_ids = [str(sample_id) for sample_id in run_sample_ids]
    row_sample_ids = [sample_id for sample_id in run_sample_ids if sample_id in table_df.index]
    col_sample_ids = [sample_id for sample_id in run_sample_ids if sample_id in table_df.columns]

    print(
        f"Feature table overlap: {len(row_sample_ids)} sample IDs in rows, "
        f"{len(col_sample_ids)} sample IDs in columns."
    )

    if not row_sample_ids and not col_sample_ids:
        raise ValueError(
            "No manifest sample IDs were found in the feature table."
        )

    if len(row_sample_ids) >= len(col_sample_ids):
        print("Detected sample IDs in table rows; transposing for decontam export.")
        run_table_df = table_df.loc[row_sample_ids, :].T.copy()
        sample_ids_in_table = row_sample_ids
    else:
        print("Detected sample IDs in table columns; using table orientation as-is for decontam export.")
        run_table_df = table_df.loc[:, col_sample_ids].copy()
        sample_ids_in_table = col_sample_ids

    run_table_df.index = run_table_df.index.astype(str)
    run_table_df.index.name = "feature_id"
    run_table_df.columns = [str(sample_id) for sample_id in run_table_df.columns]

    return run_table_df, sample_ids_in_table


# Helper: Read sample IDs from manifest

def _read_sample_ids(manifest_file):
    manifest = pd.read_csv(manifest_file, sep=None, engine="python")

    manifest.columns = [column.strip() for column in manifest.columns]

    if "sample-id" not in manifest.columns:
        raise ValueError(
            f"Expected a 'sample-id' column in {manifest_file}."
        )

    return manifest["sample-id"].astype(str).tolist()

# Helper: Load DNA concentration lookup table

def _load_dna_con_lookup():

    if not dna_con_path.exists():
        print("WARNING: DNA concentration file not found.")
        return None

    dna_con = pd.read_csv(dna_con_path)

    dna_con.columns = [column.strip() for column in dna_con.columns]

    id_candidates = [
        "Li_ID",
        "Library_ID",
        "library_id"
    ]

    conc_candidates = [
        "DNA_Con",
        "conc",
        "Concentration"
    ]

    id_col = next(
        (c for c in id_candidates if c in dna_con.columns),
        None
    )

    conc_col = next(
        (c for c in conc_candidates if c in dna_con.columns),
        None
    )

    if id_col is None or conc_col is None:
        print("WARNING: Could not identify DNA concentration columns.")
        return None

    dna_lookup = dna_con[[id_col, conc_col]].copy()

    dna_lookup = dna_lookup.rename(
        columns={
            id_col: "library_id",
            conc_col: "conc"
        }
    )

    dna_lookup["library_id"] = dna_lookup["library_id"].astype(str)

    dna_lookup["conc"] = pd.to_numeric(
        dna_lookup["conc"],
        errors="coerce"
    )

    print(f"Loaded DNA concentrations for {len(dna_lookup)} samples.")

    return dna_lookup

# Helper: Read negative controls

def _read_control_rows():

    controls = pd.read_csv(negative_controls_path)

    controls.columns = [
        column.strip()
        for column in controls.columns
    ]

    required_columns = {
        "File_Root_Name",
        "Library_ID"
    }

    missing_columns = sorted(
        required_columns - set(controls.columns)
    )

    if missing_columns:
        raise ValueError(
            f"Negative controls file missing columns: "
            f"{', '.join(missing_columns)}"
        )

    # Determine which rows are negative controls
   

    if "Is_Neg" in controls.columns:

        neg_mask = (
            controls["Is_Neg"]
            .astype(str)
            .str.upper()
            .isin({"Y", "TRUE", "T", "1"})
        )

    else:

        print(
            "WARNING: No Is_Neg column found. "
            "Assuming all rows are controls."
        )

        neg_mask = pd.Series(True, index=controls.index)

    control_rows = controls.loc[
        neg_mask,
        ["File_Root_Name", "Library_ID"]
    ].copy()

    control_rows = control_rows.rename(
        columns={
            "File_Root_Name": "sample_id",
            "Library_ID": "library_id"
        }
    )

    control_rows["sample_id"] = (
        control_rows["sample_id"].astype(str)
    )

    control_rows["library_id"] = (
        control_rows["library_id"].astype(str)
    )

    control_rows["is_neg"] = True

    control_rows["conc"] = np.nan

    # Merge DNA concentrations

    dna_lookup = _load_dna_con_lookup()

    if dna_lookup is not None:

        control_rows = control_rows.merge(
            dna_lookup,
            on="library_id",
            how="left",
            suffixes=("", "_dna")
        )

        control_rows["conc"] = (
            control_rows["conc_dna"]
            .combine_first(control_rows["conc"])
        )

        control_rows = control_rows.drop(
            columns=["conc_dna"]
        )

    control_rows["conc"] = pd.to_numeric(
        control_rows["conc"],
        errors="coerce"
    )

    print(
        f"Loaded {len(control_rows)} negative controls."
    )

    return control_rows[
        ["sample_id", "is_neg", "conc"]
    ].copy()


# Build decontam metadata

def _build_decontam_metadata(
    run_sample_ids,
    run_dir
):

    metadata = pd.read_csv(
        metadata_path,
        sep="\t"
    )

    metadata.columns = [
        column.strip()
        for column in metadata.columns
    ]

    if "sample-id" not in metadata.columns:
        raise ValueError(
            f"Expected a 'sample-id' column in "
            f"{metadata_path}."
        )


    # Keep only samples in current run

    sample_rows = metadata.loc[
        metadata["sample-id"]
        .astype(str)
        .isin(run_sample_ids)
    ].copy()

    sample_rows = sample_rows.rename(
        columns={"sample-id": "sample_id"}
    )

    sample_rows["sample_id"] = (
        sample_rows["sample_id"].astype(str)
    )

    sample_rows["is_neg"] = False

    sample_rows["conc"] = np.nan

    # Merge DNA concentrations

    dna_lookup = _load_dna_con_lookup()

    sample_library_id_col = next(
        (
            c for c in [
                "Library_ID",
                "library_id",
                "Li_ID"
            ]
            if c in sample_rows.columns
        ),
        None
    )

    if dna_lookup is not None and sample_library_id_col is not None:

        sample_rows["library_id"] = (
            sample_rows[sample_library_id_col]
            .astype(str)
        )

        sample_rows = sample_rows.merge(
            dna_lookup,
            on="library_id",
            how="left",
            suffixes=("", "_dna")
        )

        sample_rows["conc"] = (
            sample_rows["conc_dna"]
            .combine_first(sample_rows["conc"])
        )

        sample_rows = sample_rows.drop(
            columns=["conc_dna"]
        )

    # Keep required columns only

    sample_rows = sample_rows[
        ["sample_id", "is_neg", "conc"]
    ]

    # Load controls

    control_rows = _read_control_rows()

    print(
        f"Negative controls before filtering: "
        f"{len(control_rows)}"
    )

    # Combine

    combined = pd.concat(
        [sample_rows, control_rows],
        ignore_index=True
    )

    combined["sample_id"] = (
        combined["sample_id"].astype(str)
    )

    combined["is_neg"] = (
        combined["is_neg"].astype(bool)
    )

    combined["conc"] = pd.to_numeric(
        combined["conc"],
        errors="coerce"
    )

    combined = combined.drop_duplicates(
        subset=["sample_id"],
        keep="first"
    )

    combined = combined.rename(
        columns={"sample_id": "sample-id"}
    )

    metadata_path_out = (
        run_dir / "decontam_metadata.tsv"
    )

    combined.to_csv(
        metadata_path_out,
        sep="\t",
        index=False
    )

    print(
        f"Saved decontam metadata: "
        f"{metadata_path_out}"
    )

    return metadata_path_out, combined["sample-id"].tolist()


# Main decontam pipeline

def run_decontam_pipeline(
    run_label,
    run_manifest_path,
    run_dir,
    filtered_table_qza,
    filtered_rep_seqs_qza
):

    print(f"\n=== RUNNING DECONTAM: {run_label} ===\n")

    # Check required files

    if not table_path.exists():
        raise FileNotFoundError(
            f"Feature table not found at {table_path}"
        )

    if not rep_seqs_path.exists():
        raise FileNotFoundError(
            f"Rep seqs not found at {rep_seqs_path}"
        )

    # Create output directory

    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Remove old outputs

    cleanup_paths = [
        filtered_table_qza,
        filtered_rep_seqs_qza,
        run_dir / "feature_table.tsv",
        run_dir / "features_to_keep.tsv",
        run_dir / "contaminant_features.tsv",
        run_dir / "rep_seqs_decontam.fasta",
        run_dir / "decontam_metadata.tsv",
        run_dir / f"{run_label}_decontam.R",
    ]

    for path in cleanup_paths:
        if path.exists():
            path.unlink()

    # Rep seq export directory

    rep_export_dir = run_dir / "rep_seqs_export"

    shutil.rmtree(
        rep_export_dir,
        ignore_errors=True
    )

    rep_export_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    # Load sample IDs

    run_sample_ids = _read_sample_ids(
        run_manifest_path
    )

    print(
        f"Manifest contains "
        f"{len(run_sample_ids)} samples."
    )

    # Load feature table

    table_artifact = qiime2.Artifact.load(
        str(table_path)
    )

    table_df = table_artifact.view(pd.DataFrame)

    # Orient the feature table for decontam export

    run_table_df, sample_ids_in_table = _orient_feature_table_df(
        table_df,
        run_sample_ids
    )

    # Build metadata

    metadata_path_out, metadata_sample_ids = (
        _build_decontam_metadata(
            sample_ids_in_table,
            run_dir
        )
    )


    # Keep only metadata samples

    run_table_df = run_table_df.loc[
        :,
        [
            sample_id
            for sample_id in metadata_sample_ids
            if sample_id in run_table_df.columns
        ]
    ]

    # Remove duplicated ASVs

    run_table_df = run_table_df.loc[
        ~run_table_df.index.duplicated(
            keep="first"
        )
    ]

    # Save table TSV

    run_table_path = (
        run_dir / "feature_table.tsv"
    )

    run_table_df.reset_index().to_csv(
        run_table_path,
        sep="\t",
        index=False
    )

    # Build R script

    r_script_path = (
        run_dir / f"{run_label}_decontam.R"
    )

    contaminants_path = (
        run_dir / "contaminant_features.tsv"
    )

    r_script_path.write_text(
        textwrap.dedent(
            f"""
            suppressPackageStartupMessages({{
              library(phyloseq)
              library(decontam)
              library(readr)
              library(tibble)
            }})

            table_path <- {run_table_path.as_posix()!r}
            metadata_path <- {metadata_path_out.as_posix()!r}
            output_path <- {contaminants_path.as_posix()!r}

            table <- readr::read_tsv(
              table_path,
              show_col_types = FALSE
            )

            table <- tibble::column_to_rownames(
              table,
              "feature_id"
            )

            table <- as.matrix(table)

            metadata <- readr::read_tsv(
              metadata_path,
              show_col_types = FALSE
            )

            metadata <- as.data.frame(metadata)

            rownames(metadata) <- metadata[["sample-id"]]

            metadata$is_neg <- as.logical(metadata$is_neg)

            metadata$conc <- suppressWarnings(
              as.numeric(metadata$conc)
            )

            sample_ids_in_table <- intersect(
              colnames(table),
              rownames(metadata)
            )

            missing_sample_ids <- setdiff(
              rownames(metadata),
              sample_ids_in_table
            )

            if (length(missing_sample_ids) > 0) {{
              message(
                "Dropping metadata rows not present in the feature table: ",
                paste(head(missing_sample_ids, 10), collapse = ", ")
              )
            }}

            if (length(sample_ids_in_table) == 0) {{
              stop("No metadata sample IDs were found in the feature table.")
            }}

            metadata <- metadata[sample_ids_in_table, , drop = FALSE]
            table <- table[, sample_ids_in_table, drop = FALSE]

            print(summary(metadata$conc))

            ps <- phyloseq::phyloseq(
              phyloseq::otu_table(
                table,
                taxa_are_rows = TRUE
              ),
              phyloseq::sample_data(metadata)
            )

            contaminant_ids <- character(0)

            if (any(metadata$is_neg, na.rm = TRUE)) {{
              prevalence <- decontam::isContaminant(
                ps,
                method = "prevalence",
                neg = "is_neg"
              )

              contaminant_ids <- rownames(prevalence)[
                prevalence$contaminant
              ]
            }} else {{
              message("No negative controls remain after alignment; skipping prevalence-based decontam.")
            }}

            metadata_bio <- metadata[metadata$is_neg == FALSE, , drop = FALSE]
            metadata_bio <- metadata_bio[!is.na(metadata_bio$conc) & metadata_bio$conc > 0, , drop = FALSE]

            if (nrow(metadata_bio) > 0) {{
              table_bio <- table[, rownames(metadata_bio), drop = FALSE]

              ps_bio <- phyloseq::phyloseq(
                phyloseq::otu_table(
                  table_bio,
                  taxa_are_rows = TRUE
                ),
                phyloseq::sample_data(metadata_bio)
              )

              frequency <- decontam::isContaminant(
                ps_bio,
                method = "frequency",
                conc = "conc"
              )

              contaminant_ids <- union(
                contaminant_ids,
                rownames(frequency)[frequency$contaminant]
              )
            }} else {{
              message("No biological samples with positive concentrations remain after alignment; skipping frequency-based decontam.")
            }}

            readr::write_tsv(
              data.frame(feature_id = contaminant_ids),
              output_path
            )
            """
        ).strip(),
        encoding="utf-8"
    )

    # Run decontam

    print("\nRunning decontam in R...\n")

    subprocess.run(
        ["Rscript", str(r_script_path)],
        check=True
    )

    # Load contaminants

    contaminants_df = pd.read_csv(
        contaminants_path,
        sep="\t"
    )

    contaminant_ids = set(
        contaminants_df.get(
            "feature_id",
            pd.Series(dtype=str)
        )
        .dropna()
        .astype(str)
        .tolist()
    )

    print(
        f"\nIdentified "
        f"{len(contaminant_ids)} contaminant ASVs."
    )

    print(
        f"Percent contaminants: "
        f"{100 * len(contaminant_ids) / len(run_table_df):.2f}%"
    )

    # Remove contaminants

    filtered_table_df = run_table_df.loc[
        ~run_table_df.index.astype(str)
        .isin(contaminant_ids)
    ].copy()

    features_to_keep = (
        filtered_table_df.index.astype(str)
        .tolist()
    )

    if not features_to_keep:
        raise ValueError(
            "All features flagged as contaminants."
        )

    # Save feature list

    features_to_keep_path = (
        run_dir / "features_to_keep.tsv"
    )

    pd.DataFrame(
        {"feature-id": features_to_keep}
    ).to_csv(
        features_to_keep_path,
        sep="\t",
        index=False
    )

    # Filter QIIME2 feature table

    filtered_table = (
        feature_table.methods.filter_features(
            table=table_artifact,
            metadata=qiime2.Metadata.load(
                str(features_to_keep_path)
            ),
        ).filtered_table
    )

    filtered_table.save(
        str(filtered_table_qza)
    )

    # Export rep seqs

    rep_artifact = qiime2.Artifact.load(
        str(rep_seqs_path)
    )

    rep_artifact.export_data(
        str(rep_export_dir)
    )

    fasta_candidates = sorted(
        rep_export_dir.glob("*.fasta")
    )

    if not fasta_candidates:
        raise FileNotFoundError(
            f"No FASTA exported to "
            f"{rep_export_dir}"
        )

    raw_fasta_path = fasta_candidates[0]

    # Filter FASTA

    filtered_fasta_path = (
        run_dir / "rep_seqs_decontam.fasta"
    )

    with open(raw_fasta_path, "r") as fin, \
         open(filtered_fasta_path, "w") as fout:

        keep_sequence = False

        for line in fin:

            if line.startswith(">"):

                sequence_id = (
                    line[1:]
                    .strip()
                    .split()[0]
                )

                keep_sequence = (
                    sequence_id in features_to_keep
                )

                if keep_sequence:
                    fout.write(line)

            elif keep_sequence:
                fout.write(line)

    # Rebuild rep seqs artifact

    filtered_rep_seqs = qiime2.Artifact.import_data(
        "FeatureData[Sequence]",
        str(filtered_fasta_path)
    )

    filtered_rep_seqs.save(
        str(filtered_rep_seqs_qza)
    )

    # Update downstream globals

    global rep_seqs_for_classification
    global table_for_downstream

    rep_seqs_for_classification = (
        filtered_rep_seqs_qza
    )

    table_for_downstream = (
        filtered_table_qza
    )

    # Final reporting

    print("\n=== DECONTAM COMPLETE ===\n")

    print(
        f"Filtered table:\n"
        f"{filtered_table_qza}"
    )

    print(
        f"\nFiltered rep seqs:\n"
        f"{filtered_rep_seqs_qza}"
    )

    print(
        "\nDownstream analyses will use "
        "decontam-filtered outputs."
    )

### 5A. Three-Sample Decontam Test

Run this first to confirm the decontam metadata builds correctly for the three-sample subset plus the blank controls.

In [34]:
# Check if outputs already exist
if decontam_test_filtered_table_qza.exists() and decontam_test_filtered_rep_seqs_qza.exists():
    print("✓ Three-sample decontam outputs already exist, skipping...")
    print(f"  - Feature table: {decontam_test_filtered_table_qza}")
    print(f"  - Rep seqs: {decontam_test_filtered_rep_seqs_qza}")
else:
    print("Running decontam on the three-sample test set...")
    run_decontam_pipeline(
        run_label="three_samples",
        run_manifest_path=three_sample_manifest_path,
        run_dir=decontam_test_dir,
        filtered_table_qza=decontam_test_filtered_table_qza,
        filtered_rep_seqs_qza=decontam_test_filtered_rep_seqs_qza,
    )

Running decontam on the three-sample test set...
three_samples: skipping samples not present in the feature table: ['Li49677-BLANK-5a', 'Li49678-BLANK-5b', 'Li49680-BLANK-6a', 'Li49681-BLANK-6b', 'Li49690-BLANK-7a', 'Li49691-BLANK-7b', 'Li49870-BLANK-8a', 'Li49871-BLANK-8b', 'Li49592-CTR-15-24', 'Li49593-CTR-19-24']


Warning messages:
1: In .is_contaminant(seqtab, conc = conc, neg = neg, method = method,  :
  Some batches have very few (<=4) samples.
2: In .is_contaminant(seqtab, conc = conc, neg = neg, method = method,  :
  Some batches have very few (<=4) samples.
Warning messages:
1: In .is_contaminant(seqtab, conc = conc, neg = neg, method = method,  :
  Some batches have very few (<=4) samples.
2: In .is_contaminant(seqtab, conc = conc, neg = neg, method = method,  :
  Some batches have very few (<=4) samples.


three_samples: removed 770 contaminant features.
three_samples: saved filtered feature table to /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples/feature_table_decontam.qza
three_samples: saved filtered representative sequences to /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples/rep_seqs_decontam.qza
three_samples: downstream classification will use the decontam-filtered outputs.


### 5B. Full Decontam Run

Run this after the test block when you are ready to process the complete sample set. **Stop:** Do not run until explicitly confirmed.

In [38]:
from pathlib import Path

import pandas as pd
import qiime2

# Prefer the combined manifest if it exists (includes blanks)
combined_manifest = work_dir / "manifest_combined_fixed.tsv"
manifest_file = None
if combined_manifest.exists():
    manifest_file = combined_manifest
else:
    manifest_file = full_manifest_path if "full_manifest_path" in globals() else three_sample_manifest_path

# Always use the original table path (do not use decontam-filtered artifact)
table_file = table_path

print(f"Using manifest: {manifest_file}")
print(f"Using feature table: {table_file}")

if not manifest_file.exists():
    raise FileNotFoundError(f"Manifest file not found: {manifest_file}")

if not table_file.exists():
    raise FileNotFoundError(f"Feature table not found: {table_file}")


def _read_sample_ids(manifest_path):
    manifest = pd.read_csv(manifest_path, sep=None, engine="python")
    manifest.columns = [str(column).strip() for column in manifest.columns]

    if "sample-id" not in manifest.columns:
        sample_id_column = next(
            (column for column in manifest.columns if column.lower().replace("_", "-") == "sample-id"),
            None,
        )
        if sample_id_column is None:
            raise ValueError(
                f"Expected a 'sample-id' column in {manifest_path}. Found: {manifest.columns.tolist()}"
            )
        manifest = manifest.rename(columns={sample_id_column: "sample-id"})

    return manifest["sample-id"].astype(str).tolist()


run_sample_ids = _read_sample_ids(manifest_file)

table_df = qiime2.Artifact.load(str(table_file)).view(pd.DataFrame)
table_df.index = table_df.index.astype(str)
table_df.columns = table_df.columns.astype(str)

run_sample_id_set = set(run_sample_ids)
row_sample_ids = sorted(run_sample_id_set.intersection(table_df.index))
col_sample_ids = sorted(run_sample_id_set.intersection(table_df.columns))

print("\n===== MANIFEST SAMPLE IDS =====")
print(run_sample_ids[:10])

print("\n===== FEATURE TABLE ROW SAMPLE IDS =====")
print(table_df.index.tolist()[:10])
print("\n===== FEATURE TABLE COLUMN SAMPLE IDS =====")
print(table_df.columns.tolist()[:10])

print("\nManifest count:", len(run_sample_ids))
print("Feature table rows:", len(table_df.index))
print("Feature table columns:", len(table_df.columns))
print("Sample IDs found in rows:", len(row_sample_ids))
print("Sample IDs found in columns:", len(col_sample_ids))

if len(row_sample_ids) >= len(col_sample_ids):
    print("Detected sample orientation: rows")
    print("Samples missing from row-oriented table:", sorted(run_sample_id_set - set(table_df.index))[:10])
else:
    print("Detected sample orientation: columns")
    print("Samples missing from column-oriented table:", sorted(run_sample_id_set - set(table_df.columns))[:10])


Using manifest: /Users/clarence/Savolainen Lab/Microbiome/Qiime/manifest_combined_fixed.tsv
Using feature table: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/feature_table.qza

===== MANIFEST SAMPLE IDS =====
['Li49592-CTR-15-24', 'Li49593-CTR-19-24', 'Li49677-BLANK-5a', 'Li49678-BLANK-5b', 'Li49680-BLANK-6a', 'Li49681-BLANK-6b', 'Li49683-2023089-RT', 'Li49689-023055-RT', 'Li49690-BLANK-7a', 'Li49691-BLANK-7b']

===== FEATURE TABLE ROW SAMPLE IDS =====
['Li49592-CTR-15-24', 'Li49593-CTR-19-24', 'Li49677-BLANK-5a', 'Li49678-BLANK-5b', 'Li49680-BLANK-6a', 'Li49681-BLANK-6b', 'Li49683-2023089-RT', 'Li49689-023055-RT', 'Li49690-BLANK-7a', 'Li49691-BLANK-7b']

===== FEATURE TABLE COLUMN SAMPLE IDS =====
['53ea45fa55c4742b145c99a3a8e3b907', '7186328edea5572bad12f34a92d17478', '24f3f73d035f37462700bf23b25664d6', 'b5632e88d16c463a007ecb6b1a2a57c5', '27c53b43ca9d58da516d3b17cf0996f7', 'dcb41176ab63f7f28968acff18547986', '2e8f5d75b1791b6c1832b50d6e533836', '78f5a67b241650bda39a2fc51b3e

In [39]:
# Use combined manifest if available for the full decontam run
combined_manifest = work_dir / "manifest_combined_fixed.tsv"
run_manifest_path = combined_manifest if combined_manifest.exists() else full_manifest_path

if decontam_full_filtered_table_qza.exists() and decontam_full_filtered_rep_seqs_qza.exists():
    print("✓ Full decontam outputs already exist, skipping...")
    print(f"  - Feature table: {decontam_full_filtered_table_qza}")
    print(f"  - Rep seqs: {decontam_full_filtered_rep_seqs_qza}")
else:
    print("⚠ Full decontam not yet run. Starting full decontam pipeline now...")
    print(f"Using manifest for full run: {run_manifest_path}")
    run_decontam_pipeline(
        run_label='full',
        run_manifest_path=run_manifest_path,
        run_dir=decontam_full_dir,
        filtered_table_qza=decontam_full_filtered_table_qza,
        filtered_rep_seqs_qza=decontam_full_filtered_rep_seqs_qza,
    )
    print("✓ Full decontam pipeline finished.")


⚠ Full decontam not yet run. Starting full decontam pipeline now...
Using manifest for full run: /Users/clarence/Savolainen Lab/Microbiome/Qiime/manifest_combined_fixed.tsv

=== RUNNING DECONTAM: full ===

Manifest contains 49 samples.
Feature table overlap: 49 sample IDs in rows, 0 sample IDs in columns.
Detected sample IDs in table rows; transposing for decontam export.
Loaded DNA concentrations for 137 samples.
Loaded DNA concentrations for 137 samples.
Loaded 10 negative controls.
Negative controls before filtering: 10
Saved decontam metadata: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full/decontam_metadata.tsv

Running decontam in R...

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max.    NA's 
  0.010   1.645   3.110   4.782   6.930  17.020      10 

Identified 1792 contaminant ASVs.
Percent contaminants: 0.56%

=== DECONTAM COMPLETE ===

Filtered table:
/Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full/feature_table_decontam.qza

Filtered r

In [ ]:

# Verify decontam output and understand what happened

import pandas as pd

print("\n=== DECONTAM VERIFICATION ===\n")

# Load decontam metadata
metadata_df = pd.read_csv(decontam_full_metadata_path, sep="\t")

print("Decontam Metadata Summary:")
print(f"  Total samples in metadata: {len(metadata_df)}")
print(f"  Biological samples (is_neg=False): {(metadata_df['is_neg'] == False).sum()}")
print(f"  Negative control samples (is_neg=True): {(metadata_df['is_neg'] == True).sum()}")

print("\nConcentration (DNA_Con) values:")
print(f"  Samples with concentration data: {metadata_df['conc'].notna().sum()}")
print(f"  Samples with NaN concentration: {metadata_df['conc'].isna().sum()}")

conc_values = metadata_df[metadata_df['conc'].notna()]['conc']
if len(conc_values) > 0:
    print(f"  Min concentration: {conc_values.min():.3f}")
    print(f"  Max concentration: {conc_values.max():.3f}")
    print(f"  Mean concentration: {conc_values.mean():.3f}")
    positive_conc = (conc_values > 0).sum()
    print(f"  Strictly positive (>0): {positive_conc} out of {len(conc_values)}")

# Load contaminant features
contaminants_df = pd.read_csv(decontam_full_contaminants_path, sep="\t")
print(f"\nContaminant Features Identified:")
print(f"  Total features flagged: {len(contaminants_df)}")

# Load the original and post-decontam feature counts
feature_table_full = decontam_full_dir / "feature_table.tsv"
if feature_table_full.exists():
    ft_df = pd.read_csv(feature_table_full, sep="\t", index_col="feature_id")
    print(f"  Total features in pre-decontam table: {len(ft_df)}")
    print(f"  Features remaining after decontam: {len(ft_df) - len(contaminants_df)}")
    print(f"  Percent removed: {100 * len(contaminants_df) / len(ft_df):.2f}%")

print("\nWhy frequency-based decontam was skipped:")
print(f"  - Reason: No strictly positive concentrations in the metadata after alignment")
print(f"  - The prevalence-based method (using negative controls) was still applied")
print(f"  - This means only features overrepresented in negative controls were flagged")



=== DECONTAM VERIFICATION ===

Decontam Metadata Summary:
  Total samples in metadata: 49
  Biological samples (is_neg=False): 39
  Negative control samples (is_neg=True): 10

Concentration (DNA_Con) values:
  Samples with concentration data: 39
  Samples with NaN concentration: 10
  Min concentration: 0.000
  Max concentration: 17.020
  Mean concentration: 4.782
  Strictly positive (>0): 37 out of 39

Contaminant Features Identified:
  Total features flagged: 965
  Total features in pre-decontam table: 317921
  Features remaining after decontam: 316956
  Percent removed: 0.30%

Why frequency-based decontam was skipped:
  - Reason: No strictly positive concentrations in the metadata after alignment
  - The prevalence-based method (using negative controls) was still applied
  - This means only features overrepresented in negative controls were flagged


## 6. Classify ASVs with the Trained SILVA Classifier

Load the trained classifier and apply it to representative sequences, split into separate runs for the three-sample test and the full sample set.

### 6A. Three-Sample Classification

Classify representative sequences from the three-sample decontam run.

In [35]:
from qiime2.plugins import feature_classifier

try:
    classifier_path
except NameError:
    raise NameError("Variable 'classifier_path' is not defined. Please run Section 1 (Paths and Inputs) first.")

print("Classifying three-sample decontam run...")
if not decontam_test_filtered_rep_seqs_qza.exists():
    raise FileNotFoundError("Three-sample rep seqs not found. Run section 5A (decontam test) first.")

if not classifier_path.exists():
    raise FileNotFoundError(f"Classifier not found at {classifier_path}. Please run Section 4 to train it first.")

classifier = qiime2.Artifact.load(str(classifier_path))
rep_seqs = qiime2.Artifact.load(str(decontam_test_filtered_rep_seqs_qza))

print("Classifying representative sequences...")
taxonomy = feature_classifier.methods.classify_sklearn(
    reads=rep_seqs,
    classifier=classifier
).classification

taxonomy.save(str(decontam_test_taxonomy_out))
print(f"Saved three-sample taxonomy: {decontam_test_taxonomy_out}")


Classifying three-sample decontam run...
Classifying representative sequences...
Saved three-sample taxonomy: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples/taxonomy_silva.qza


### 6B. Full Sample Classification

Classify representative sequences from the full sample decontam run.

In [ ]:
from qiime2.plugins import feature_classifier

print("Classifying full sample decontam run...")
if not decontam_full_filtered_rep_seqs_qza.exists():
    raise FileNotFoundError("Full sample rep seqs not found. Run section 5B (decontam full) first.")

if not classifier_path.exists():
    raise FileNotFoundError(f"Classifier not found at {classifier_path}. Please run Section 4 to train it first.")

classifier = qiime2.Artifact.load(str(classifier_path))
rep_seqs = qiime2.Artifact.load(str(decontam_full_filtered_rep_seqs_qza))

print("Classifying representative sequences...")
taxonomy = feature_classifier.methods.classify_sklearn(
    reads=rep_seqs,
    classifier=classifier
).classification

taxonomy.save(str(decontam_full_taxonomy_out))
print(f"Saved full sample taxonomy: {decontam_full_taxonomy_out}")
 
## 7. Build Taxonomy Barplot and Export Results

Skip this and use the R script in R instead.
<VSCode.Cell id="#VSC-66a02007" language="markdown">
## 8. Build Phylogenetic Trees

Create phylogenetic trees from the decontam-filtered representative sequences using MAFFT alignment, masking, and FastTree. This produces both unrooted and rooted trees for downstream phylogenetic analysis.


Classifying full sample decontam run...
Classifying representative sequences...
Saved full sample taxonomy: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full/taxonomy_silva.qza


### 8A. Three-Sample Phylogenetic Tree

Build a phylogenetic tree from three-sample decontam representative sequences using MAFFT alignment and FastTree.


In [ ]:
from qiime2.plugins import alignment, phylogeny
import pandas as pd

if not decontam_test_rooted_tree.exists():
    print("Building three-sample phylogenetic tree...")
    if not decontam_test_filtered_rep_seqs_qza.exists():
        raise FileNotFoundError("Three-sample rep seqs not found. Run section 5A (decontam test) first.")

    # Load metadata and keep only biological samples (remove blanks/negative controls)
    if not decontam_test_metadata_path.exists():
        raise FileNotFoundError("Three-sample decontam metadata not found. Run section 5A (decontam test) first.")

    metadata_df = pd.read_csv(decontam_test_metadata_path, sep="\t")
    metadata_df.columns = [str(c).strip() for c in metadata_df.columns]

    if "sample-id" not in metadata_df.columns:
        raise ValueError("Expected 'sample-id' column in three-sample decontam metadata.")

    metadata_df["sample-id"] = metadata_df["sample-id"].astype(str)

    if "is_neg" in metadata_df.columns:
        is_neg = metadata_df["is_neg"].astype(str).str.strip().str.lower().isin({"true", "t", "1", "y", "yes"})
        bio_sample_ids = metadata_df.loc[~is_neg, "sample-id"].tolist()
    else:
        print("  - WARNING: 'is_neg' missing in metadata; using all samples for tree building.")
        bio_sample_ids = metadata_df["sample-id"].tolist()

    # Load rep seq IDs first so we can orient the feature table correctly
    rep_seqs_artifact = qiime2.Artifact.load(str(decontam_test_filtered_rep_seqs_qza))
    rep_seqs_df = rep_seqs_artifact.view(pd.Series)
    rep_seq_ids = set(rep_seqs_df.index.astype(str))

    # Load feature table and detect whether ASVs are in rows or columns
    feature_table = qiime2.Artifact.load(str(decontam_test_filtered_table_qza))
    feature_table_df = feature_table.view(pd.DataFrame)
    feature_table_df.index = feature_table_df.index.astype(str)
    feature_table_df.columns = feature_table_df.columns.astype(str)

    row_asv_overlap = len(set(feature_table_df.index) & rep_seq_ids)
    col_asv_overlap = len(set(feature_table_df.columns) & rep_seq_ids)

    if col_asv_overlap > row_asv_overlap:
        print("  - Detected ASVs in columns; transposing feature table for filtering.")
        feature_table_df = feature_table_df.T

    print(f"  - Table orientation normalized to ASVs x samples: {len(feature_table_df.index)} ASVs x {len(feature_table_df.columns)} samples")

    # Keep only biological samples in the table
    bio_sample_ids_in_table = [sample_id for sample_id in bio_sample_ids if sample_id in feature_table_df.columns]
    if not bio_sample_ids_in_table:
        raise ValueError("No biological samples from metadata were found in the feature table.")

    feature_table_df = feature_table_df.loc[:, bio_sample_ids_in_table]
    print(f"  - Biological samples retained for tree: {len(bio_sample_ids_in_table)}")

    # Calculate prevalence (# samples with >0 abundance) and total abundance per ASV
    prevalence = (feature_table_df > 0).sum(axis=1)
    total_abundance = feature_table_df.sum(axis=1)

    # Filter ASVs: keep those in 2+ samples with total abundance >= 10
    asv_mask = (prevalence >= 2) & (total_abundance >= 10)
    filtered_asv_ids = asv_mask[asv_mask].index.tolist()

    print(f"  - ASV filtering: {len(filtered_asv_ids)} of {len(feature_table_df)} ASVs meet criteria (>=2 samples, >=10 abundance)")

    # Create filtered rep seqs as FASTA
    filtered_rep_seqs_path = decontam_test_dir / "rep_seqs_filtered_for_tree.fasta"
    with open(filtered_rep_seqs_path, "w") as fout:
        for asv_id in filtered_asv_ids:
            if asv_id in rep_seqs_df.index:
                seq = str(rep_seqs_df[asv_id])
                fout.write(f">{asv_id}\n{seq}\n")

    # Import filtered rep seqs as QIIME2 artifact
    rep_seqs = qiime2.Artifact.import_data(
        "FeatureData[Sequence]",
        str(filtered_rep_seqs_path)
    )

    print("  - Aligning sequences with MAFFT...")
    aligned_seqs = alignment.methods.mafft(
        sequences=rep_seqs
    ).alignment
    aligned_seqs.save(str(decontam_test_aligned_seqs))

    print("  - Masking alignment...")
    masked_aligned_seqs = alignment.methods.mask(
        alignment=aligned_seqs
    ).masked_alignment
    masked_aligned_seqs.save(str(decontam_test_masked_aligned_seqs))

    print("  - Building unrooted tree with FastTree...")
    unrooted_tree = phylogeny.methods.fasttree(
        alignment=masked_aligned_seqs
    ).tree
    unrooted_tree.save(str(decontam_test_unrooted_tree))

    print("  - Rooting tree at midpoint...")
    rooted_tree = phylogeny.methods.midpoint_root(
        tree=unrooted_tree
    ).rooted_tree
    rooted_tree.save(str(decontam_test_rooted_tree))

    print(f"✓ Three-sample tree complete:")
    print(f"  - Aligned seqs: {decontam_test_aligned_seqs}")
    print(f"  - Masked aligned seqs: {decontam_test_masked_aligned_seqs}")
    print(f"  - Unrooted tree: {decontam_test_unrooted_tree}")
    print(f"  - Rooted tree: {decontam_test_rooted_tree}")
else:
    print("✓ Three-sample phylogenetic tree already exists, skipping...")
    print(f"  - Rooted tree: {decontam_test_rooted_tree}")

Building three-sample phylogenetic tree...
  - Aligning sequences with MAFFT...
Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command: mafft --preservecase --inputorder --thread 1 /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/qiime2/clarence/data/e20833b1-89b9-484e-871d-8542b204bb88/data/dna-sequences.fasta



inputfile = orig
55555 x 525 - 280 d
nthread = 1
nthreadpair = 1
nthreadtb = 1
ppenalty_ex = 0
stacksize: 8176 kb->10850 kb
generating a scoring matrix for nucleotide (dist=200) ... done
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..
 55501 / 55555 (thread    0)
done.

Constructing a UPGMA tree (efffree=0) ... 
 55550 / 55555
done.

Progressive alignment 1/2... 
STEP  55301 / 55554  h
Reallocating..done. *alloclen = 2056
STEP  55501 / 55554  h
done.

Making a distance matrix from msa.. 
 55500 / 55555 (thread    0)
done.

Constructing a UPGMA tree (efffree=1) ... 
 55550 / 55555
done.

Progressive alignment 2/2... 
STEP  55201 / 55554  h
Reallocating..done. *alloclen = 2122
STEP  55501 / 55554  h
done.

disttbfast (nuc) Version 7.526
alg=A, model=DNA200 (2), 1.53 (4.59), -0.00 (-0.00), noshift, amax=0.0
0 thread(s)


Strategy:
 FFT-NS-2 (Fast but rough)
 Progressive method (guide trees were built 2 times.)

If unsure which option to use, try 'mafft --auto input > outp

  - Masking alignment...
  - Building unrooted tree with FastTree...
Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command: FastTree -quote -nt /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/qiime2/clarence/data/17e2e316-f989-4824-97ff-3a4ecb1a10db/data/aligned-dna-sequences.fasta



FastTree Version 2.2.0 Double precision
Alignment: /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/qiime2/clarence/data/17e2e316-f989-4824-97ff-3a4ecb1a10db/data/aligned-dna-sequences.fasta
Nucleotide distances: Jukes-Cantor Joins: balanced Support: SH-like 1000
Search: Normal +NNI +SPR (2 rounds range 10) +ML-NNI opt-each=1
TopHits: 1.00*sqrtN close=default refresh=0.80
ML Model: Jukes-Cantor, CAT approximation with 20 rate categories
      0.11 seconds: Identified unique sequences
      0.72 seconds: Top hits for   1859 of   7494 seqs (at seed    100)
      0.96 seconds: Top hits for   2620 of   7494 seqs (at seed    200)
      1.16 seconds: Top hits for   3044 of   7494 seqs (at seed    500)
      1.29 seconds: Top hits for   3349 of   7494 seqs (at seed    700)
      1.43 seconds: Top hits for   3753 of   7494 seqs (at seed    900)
      1.59 seconds: Top hits for   4189 of   7494 seqs (at seed   1000)
      1.69 seconds: Top hits for   4399 of   7494 seqs (at seed   1200)
      1

  - Rooting tree at midpoint...
✓ Three-sample tree complete:
  - Aligned seqs: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples/aligned_rep_seqs.qza
  - Masked aligned seqs: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples/masked_aligned_rep_seqs.qza
  - Unrooted tree: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples/unrooted_tree.qza
  - Rooted tree: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/three_samples/rooted_tree.qza


### 8B. Full Sample Phylogenetic Tree

Build a phylogenetic tree from full sample decontam representative sequences using MAFFT alignment and FastTree.


In [ ]:
from qiime2.plugins import alignment, phylogeny
import pandas as pd

if not decontam_full_rooted_tree.exists():
    print("Building full sample phylogenetic tree...")
    if not decontam_full_filtered_rep_seqs_qza.exists():
        raise FileNotFoundError("Full sample rep seqs not found. Run section 5B (decontam full) first.")

    # Load metadata and keep only biological samples (remove blanks/negative controls)
    if not decontam_full_metadata_path.exists():
        raise FileNotFoundError("Full-sample decontam metadata not found. Run section 5B (decontam full) first.")

    metadata_df = pd.read_csv(decontam_full_metadata_path, sep="\t")
    metadata_df.columns = [str(c).strip() for c in metadata_df.columns]

    if "sample-id" not in metadata_df.columns:
        raise ValueError("Expected 'sample-id' column in full-sample decontam metadata.")

    metadata_df["sample-id"] = metadata_df["sample-id"].astype(str)

    if "is_neg" in metadata_df.columns:
        is_neg = metadata_df["is_neg"].astype(str).str.strip().str.lower().isin({"true", "t", "1", "y", "yes"})
        bio_sample_ids = metadata_df.loc[~is_neg, "sample-id"].tolist()
    else:
        print("  - WARNING: 'is_neg' missing in metadata; using all samples for tree building.")
        bio_sample_ids = metadata_df["sample-id"].tolist()

    # Load rep seq IDs first so we can orient the feature table correctly
    rep_seqs_artifact = qiime2.Artifact.load(str(decontam_full_filtered_rep_seqs_qza))
    rep_seqs_df = rep_seqs_artifact.view(pd.Series)
    rep_seq_ids = set(rep_seqs_df.index.astype(str))

    # Load feature table and detect whether ASVs are in rows or columns
    print("  - Loading feature table...")
    feature_table = qiime2.Artifact.load(str(decontam_full_filtered_table_qza))
    print("  - Converting to DataFrame...")
    feature_table_df = feature_table.view(pd.DataFrame)
    feature_table_df.index = feature_table_df.index.astype(str)
    feature_table_df.columns = feature_table_df.columns.astype(str)

    row_asv_overlap = len(set(feature_table_df.index) & rep_seq_ids)
    col_asv_overlap = len(set(feature_table_df.columns) & rep_seq_ids)

    if col_asv_overlap > row_asv_overlap:
        print("  - Detected ASVs in columns; transposing feature table for filtering.")
        feature_table_df = feature_table_df.T

    print(f"  - Table orientation normalized to ASVs x samples: {len(feature_table_df.index)} ASVs x {len(feature_table_df.columns)} samples")

    # Keep only biological samples in the table
    bio_sample_ids_in_table = [sample_id for sample_id in bio_sample_ids if sample_id in feature_table_df.columns]
    if not bio_sample_ids_in_table:
        raise ValueError("No biological samples from metadata were found in the feature table.")

    feature_table_df = feature_table_df.loc[:, bio_sample_ids_in_table]
    print(f"  - Biological samples retained for tree: {len(bio_sample_ids_in_table)}")

    # Calculate prevalence (# samples with >0 abundance) and total abundance per ASV
    print("  - Calculating prevalence and abundance...")
    prevalence = (feature_table_df > 0).sum(axis=1)
    total_abundance = feature_table_df.sum(axis=1)

    # Filter ASVs: keep those in 5+ samples (12.5% of samples) with total abundance >= 100
    asv_mask = (prevalence >= 5) & (total_abundance >= 100)
    filtered_asv_ids = asv_mask[asv_mask].index.tolist()

    print(f"  - ASV filtering: {len(filtered_asv_ids)} of {len(feature_table_df)} ASVs meet criteria (>=5 samples, >=100 abundance)")

    # Create filtered rep seqs as FASTA
    filtered_rep_seqs_path = decontam_full_dir / "rep_seqs_filtered_for_tree.fasta"
    with open(filtered_rep_seqs_path, "w") as fout:
        for asv_id in filtered_asv_ids:
            if asv_id in rep_seqs_df.index:
                seq = str(rep_seqs_df[asv_id])
                fout.write(f">{asv_id}\n{seq}\n")

    # Import filtered rep seqs as QIIME2 artifact
    rep_seqs = qiime2.Artifact.import_data(
        "FeatureData[Sequence]",
        str(filtered_rep_seqs_path)
    )

    print("  - Aligning sequences with MAFFT...")
    aligned_seqs = alignment.methods.mafft(
        sequences=rep_seqs
    ).alignment
    aligned_seqs.save(str(decontam_full_aligned_seqs))

    print("  - Masking alignment...")
    masked_aligned_seqs = alignment.methods.mask(
        alignment=aligned_seqs
    ).masked_alignment
    masked_aligned_seqs.save(str(decontam_full_masked_aligned_seqs))

    print("  - Building unrooted tree with FastTree...")
    unrooted_tree = phylogeny.methods.fasttree(
        alignment=masked_aligned_seqs
    ).tree
    unrooted_tree.save(str(decontam_full_unrooted_tree))

    print("  - Rooting tree at midpoint...")
    rooted_tree = phylogeny.methods.midpoint_root(
        tree=unrooted_tree
    ).rooted_tree
    rooted_tree.save(str(decontam_full_rooted_tree))

    print(f"✓ Full sample tree complete:")
    print(f"  - Aligned seqs: {decontam_full_aligned_seqs}")
    print(f"  - Masked aligned seqs: {decontam_full_masked_aligned_seqs}")
    print(f"  - Unrooted tree: {decontam_full_unrooted_tree}")
    print(f"  - Rooted tree: {decontam_full_rooted_tree}")
else:
    print("✓ Full sample phylogenetic tree already exists, skipping...")
    print(f"  - Rooted tree: {decontam_full_rooted_tree}")

print("\nPhylogenetic tree building complete!")

Building full sample phylogenetic tree...
  - Loading feature table...
  - Converting to DataFrame...
  - Detected ASVs in columns; transposing feature table for filtering.
  - Table orientation normalized to ASVs x samples: 316129 ASVs x 49 samples
  - Biological samples retained for tree: 39
  - Calculating prevalence and abundance...
  - ASV filtering: 44711 of 316129 ASVs meet criteria (>=5 samples, >=100 abundance)
  - Aligning sequences with MAFFT...
Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command: mafft --preservecase --inputorder --thread 1 /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/qiime2/clarence/data/42b5fedc-8b0b-4fdf-87f8-7b982fa8fe81/data/dna-sequences.fasta



inputfile = orig
44711 x 467 - 298 d
nthread = 1
nthreadpair = 1
nthreadtb = 1
ppenalty_ex = 0
stacksize: 8176 kb->8732 kb
generating a scoring matrix for nucleotide (dist=200) ... done
Gap Penalty = -1.53, +0.00, +0.00



Making a distance matrix ..
 44701 / 44711 (thread    0)
done.

Constructing a UPGMA tree (efffree=0) ... 
 44700 / 44711
done.

Progressive alignment 1/2... 
STEP  44601 / 44710 
Reallocating..done. *alloclen = 1939
STEP  44701 / 44710  h
done.

Making a distance matrix from msa.. 
 44700 / 44711 (thread    0)
done.

Constructing a UPGMA tree (efffree=1) ... 
 44700 / 44711
done.

Progressive alignment 2/2... 
STEP  44601 / 44710  h
Reallocating..done. *alloclen = 1939
STEP  44701 / 44710  h
done.

disttbfast (nuc) Version 7.526
alg=A, model=DNA200 (2), 1.53 (4.59), -0.00 (-0.00), noshift, amax=0.0
0 thread(s)


Strategy:
 FFT-NS-2 (Fast but rough)
 Progressive method (guide trees were built 2 times.)

If unsure which option to use, try 'mafft --auto input > output'

  - Masking alignment...
  - Building unrooted tree with FastTree...
Running external command line application. This may print messages to stdout and/or stderr.
The command being run is below. This command cannot be manually re-run as it will depend on temporary files that no longer exist.

Command: FastTree -quote -nt /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/qiime2/clarence/data/1dd431b7-e75d-4afe-905f-303e3b33f645/data/aligned-dna-sequences.fasta



FastTree Version 2.2.0 Double precision
Alignment: /var/folders/hm/m1vxxms91sv3lsvjp0392mg80000gn/T/qiime2/clarence/data/1dd431b7-e75d-4afe-905f-303e3b33f645/data/aligned-dna-sequences.fasta
Nucleotide distances: Jukes-Cantor Joins: balanced Support: SH-like 1000
Search: Normal +NNI +SPR (2 rounds range 10) +ML-NNI opt-each=1
TopHits: 1.00*sqrtN close=default refresh=0.80
ML Model: Jukes-Cantor, CAT approximation with 20 rate categories
      0.10 seconds: Top hits for    191 of   1928 seqs (at seed    100)
      0.21 seconds: Top hits for   1102 of   1928 seqs (at seed    900)
      0.31 seconds: Top hits for   1863 of   1928 seqs (at seed   1700)
      0.48 seconds: Joined    200 of   1925
      0.60 seconds: Joined    300 of   1925
      0.73 seconds: Joined    400 of   1925
      0.85 seconds: Joined    500 of   1925
      0.97 seconds: Joined    600 of   1925
      1.08 seconds: Joined    700 of   1925
      1.19 seconds: Joined    800 of   1925
      1.30 seconds: Joined    900 o

  - Rooting tree at midpoint...
✓ Full sample tree complete:
  - Aligned seqs: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full/aligned_rep_seqs.qza
  - Masked aligned seqs: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full/masked_aligned_rep_seqs.qza
  - Unrooted tree: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full/unrooted_tree.qza
  - Rooted tree: /Users/clarence/Savolainen Lab/Microbiome/Qiime/output/decontam/full/rooted_tree.qza

Phylogenetic tree building complete!
